<a target="_blank" href="https://colab.research.google.com/github/AshishKumar4/dew/blob/main/tutorials/03-text-to-image-with-guidance.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Text to image with classifier-free guidance

[Notebook 02](02-train-a-diffusion-model.ipynb) trained a model that knows what flowers look like but cannot be told which one. This notebook conditions the same kind of DiT on text. Each record in Oxford Flowers carries a caption like "a photo of a water lily"; a CLIP text encoder turns the caption into embeddings, and the trainer hands them to the model alongside the noise level. At sampling time the model can then be steered: classifier-free guidance extrapolates away from the unconditional prediction toward the caption, and `guidance_scale` says how hard.

The notebook also trains in the latent space of the Stable Diffusion VAE rather than in pixels: the VAE compresses each image by 8 in width and height, so the diffusion model sees 16x16x4 tensors instead of 128x128x3. That is how the large text-to-image models are trained, and here it makes the run cheaper at higher resolution. Set `USE_VAE = False` in the configuration cell to train on pixels instead; everything else stays the same.

**Expected time**: about 30 minutes on a Colab A100, about 40 minutes on a local RTX 4080, about 45 on a TPU v5e. The first run downloads the dataset, CLIP-L/14 and the SD VAE, roughly 2.5 GB in total.

In [ ]:
# On Colab: install dew and the JAX build for the runtime. Locally this cell is a no-op.
try:
    import google.colab  # noqa: F401
    import subprocess, sys
    try:
        import jax
        tpu = any("tpu" in str(d).lower() for d in jax.devices())
    except Exception:
        tpu = False
    extra = "jax[tpu]" if tpu else "jax[cuda12]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dew-ml[tfds] @ git+https://github.com/AshishKumar4/dew", extra, "matplotlib"])
except ImportError:
    pass

In [ ]:
# The whole run in one place.
BATCH_SIZE = 32           # images per optimizer step
IMAGE_SIZE = 128          # pixels; the VAE maps this to 16x16x4 latents
EPOCHS = 60               # passes over the dataset
LEARNING_RATE = 2e-4
MODEL = dict(patch_size=2, emb_features=384, num_layers=8, num_heads=6)
USE_VAE = True            # train in the SD VAE's latent space; False trains on pixels
RUN_NAME = "flowers-text" # checkpoints land in ./checkpoints/<RUN_NAME>
WANDB_PROJECT = None      # set to a project name to log the run
WORKER_COUNT = 4
PROMPTS = ("a water lily", "a sunflower", "a red rose", "a purple orchid")
GUIDANCE_SCALES = (1.0, 2.0, 4.0, 7.0)  # one row of the final grid each
SAMPLE_STEPS = 40
SEED = 0

In [ ]:
import jax

print(jax.devices())
print(jax.default_backend())

## Conditioning and the VAE

Two objects change the training contract. `CLIPTextEncoder` is a conditioning encoder: it tokenizes a caption and runs it through a frozen CLIP text model, and the trainer swaps about one row in eight (the `unconditional_prob` of 0.12) for an empty caption, so the one model learns both the conditional and the unconditional distribution that guidance needs. `StableDiffusionVAE` is the autoencoder: `encode` compresses a batch of images to latents before the diffusion loss and `decode` expands sampled latents back to images, both inside the compiled step.

With `USE_VAE = True` the model config below describes the latent grid, and the shapes follow from `DiffusionInputConfig` and the VAE's `downscale_factor` without you computing anything.

In [ ]:
from dew.data.dataloaders import get_dataset_grain

data = get_dataset_grain("oxford_flowers102", batch_size=BATCH_SIZE, image_scale=IMAGE_SIZE,
                         worker_count=WORKER_COUNT, val_count=4 * BATCH_SIZE)
steps_per_epoch = data["train_len"] // BATCH_SIZE
print(f"{data['train_len']} training records, {steps_per_epoch} steps per epoch")

## The model and its inputs

The preset is the same EDM pair as before. The model is a DiT; the changes from notebook 02 are that `patch_size` shrinks to 2 to match the smaller latent grid, and the input config carries one condition: the CLIP encoder reading the batch's tokenized captions. `model_key_override` names the keyword the model receives them under (`textcontext`).

In [ ]:
from dew.inputs import ConditionalInputConfig, DiffusionInputConfig
from dew.inputs.encoders import CLIPTextEncoder
from dew.registry import apply_precision_policy, build_model
from dew.diffusion.transforms import get_diffusion_preset

train_schedule, sample_schedule, transform = get_diffusion_preset("edm")

text_encoder = CLIPTextEncoder.from_modelname("openai/clip-vit-large-patch14")
inputs = DiffusionInputConfig(
    sample_data_key="image",
    sample_data_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    conditions=[ConditionalInputConfig(
        encoder=text_encoder,
        conditioning_data_key="text",   # the batch key the grain augmenter fills
        pretokenized=True,              # tokens arrive tokenized from the data workers
        unconditional_input="",         # the empty caption the trainer drops back to
        model_key_override="textcontext",
    )],
)

autoencoder = None
if USE_VAE:
    from dew.nn.autoencoders.sd_vae import StableDiffusionVAE
    autoencoder = StableDiffusionVAE()

channels = autoencoder.latent_channels if autoencoder is not None else 3
model = build_model("simple_dit", apply_precision_policy(
    "simple_dit", {**MODEL, "output_channels": channels},
    dtype="bfloat16", attention_impl="auto"))

## Train

The trainer call is one line longer than notebook 02's: pass `autoencoder=...` and the objective encodes on the way in while the samplers decode on the way out. Validation generates four caption-conditioned samples per epoch from the EMA weights, so the grids improve as the run goes.

In [ ]:
import optax
from dew.sampling import EulerAncestralSampler
from dew.training import ObjectiveTrainer

wandb_config = None
if WANDB_PROJECT is not None:
    wandb_config = {
        "project": WANDB_PROJECT,
        "name": RUN_NAME,
        "config": {"model": MODEL, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
                   "epochs": EPOCHS, "learning_rate": LEARNING_RATE, "use_vae": USE_VAE},
    }

trainer = ObjectiveTrainer(
    model, optax.adamw(LEARNING_RATE),
    input_config=inputs,
    noise_schedule=train_schedule,
    model_output_transform=transform,
    autoencoder=autoencoder,
    rngs=jax.random.PRNGKey(SEED),
    name=RUN_NAME,
    wandb_config=wandb_config,
    log_every=50,
)

n_params = sum(p.size for p in jax.tree_util.tree_leaves(trainer.state.params))
print(f"{n_params / 1e6:.1f}M parameters in the diffusion model")

In [ ]:
state = trainer.fit(
    data,
    training_steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    val_steps_per_epoch=1,
    sampler_class=EulerAncestralSampler,
    sampling_noise_schedule=sample_schedule,
)

## Classifier-free guidance

At sampling time the model sees two captions: the real one and the empty one. Both predictions describe the same noise level, and their difference is the direction the caption points in. Guidance scales that difference: a scale of 1 ignores the empty caption entirely, 2 to 4 is the usual range for a small model, and past about 7 the samples get saturated and lose variety.

`guidance_start` and `guidance_stop` limit the extrapolation to an interval of the trajectory, as a fraction of the way from noise (0) to image (1). Guidance helps most in the middle, where structure forms; leaving the last few percent unguided keeps colors from blowing out.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from dew.sampling import EulerAncestralSampler

def save_grid(rows, path):
    '''rows: list of [n, H, W, 3] float arrays in [-1, 1]; one grid row per entry.'''
    frames = [np.clip((np.asarray(r) + 1) * 127.5, 0, 255).astype(np.uint8) for r in rows]
    grid = np.concatenate([np.concatenate(f, axis=1) for f in frames], axis=0)
    import os
    os.makedirs(os.path.dirname(path), exist_ok=True)
    Image.fromarray(grid).save(path)
    plt.figure(figsize=(frames[0].shape[0] * 1.6, len(frames) * 1.6))
    plt.imshow(grid)
    plt.axis("off")
    plt.show()
    return path

rows = []
for scale in GUIDANCE_SCALES:
    sampler = EulerAncestralSampler(model, sample_schedule, transform, inputs,
                                    autoencoder=autoencoder,
                                    guidance_scale=scale, guidance_start=0.1, guidance_stop=0.9)
    images = sampler.generate_samples(params=state.ema_params, num_samples=len(PROMPTS),
                                      resolution=IMAGE_SIZE, diffusion_steps=SAMPLE_STEPS,
                                      conditioning=list(PROMPTS))
    rows.append(images)
    print(f"guidance_scale={scale}: done")

save_grid(rows, "samples/03-guidance-grid.png")

## Read the grid

Each row is one guidance scale, each column one caption, the same four captions for every row. At scale 1 the model answers the caption only faintly; by 4 the flowers follow their names; at 7 the colors run hot and the four columns drift toward the same look, which is the variety loss guidance trades for prompt adherence.

## Where to go next

[Notebook 04](04-samplers-and-schedules.ipynb) loads the checkpoint from notebook 02 and compares the solvers. To push this model further, raise `EPOCHS` and read the validation grids each epoch: captions start steering the shapes long before the details settle.